# Reproducibility Notice:
Deep learning models trained on GPUs exhibit inherent non-determinism due to parallel floating-point operations. Re-running this notebook from scratch will result in slight metric variations (± 1-2%) compared to the exact numbers reported in the official manuscript.

The architectural choices, the Sparse Categorical Focal Loss implementation, and the Conformal Prediction fallback logic remain perfectly consistent with the paper. The exact metrics and confusion matrix reported in our study (Global Acc: 96.48%, Flood F1-Score: 78.65%) are documented in the original_training_report.pdf provided in this repository.

In [1]:
%pip install reportlab -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 38.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


### 1. Automated Evaluation and Reporting Module

The cell below contains the code to generate the standardized PDF and CSV report files. This module compiles the deep learning model's outputs after it completes all training and evaluation steps, ensuring full traceability of the experiments.

The generated PDF report is structurally organized to display:
* **Hyperparameters:** Input size, batch size, training epochs per phase, and resampling size (samples per class).
* **General Metrics:** Global accuracy, balanced accuracy, test loss, and average inference time.
* **Class-wise Metrics:** Precision, Recall, F1-Score, and Support for each specific water level state.
* **Visual Analytics:** Training and validation learning curves, alongside the out-of-time Confusion Matrix.

*Note: This cell uses the `%%writefile` command to save the script locally as `report.py`. The function will be imported and executed at the end of the pipeline to summarize the post-fallback results.*

In [2]:
%%writefile report.py
import os
import datetime
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_fscore_support, balanced_accuracy_score
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import cm as rl_cm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image
from reportlab.lib.enums import TA_CENTER

def fig_to_image(fig, path):
    fig.savefig(path, format='png', bbox_inches='tight', dpi=150)
    return path

def gerar_relatorio(
    config_name, output_dir, class_names,
    acc, loss, tempos_ms,
    y_true, y_pred, cm,
    history, history_fine,
    img_size, batch_size, samples_per_class,
    csv_path='/kaggle/working/resultados.csv'
):
    path_curvas = os.path.join(output_dir, '_tmp_curvas.png')
    path_cm     = os.path.join(output_dir, '_tmp_cm.png')

    precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=list(range(len(class_names))))
    bal_acc = balanced_accuracy_score(y_true, y_pred)

    # CSV
    resultado = {
        'config':              config_name,
        'data':                datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
        'acuracia':            acc,
        'perda':               loss,
        'acuracia_balanceada': bal_acc,
        'tempo_medio_ms':      np.mean(tempos_ms),
        'img_size':            img_size[0],
        'batch_size':          batch_size,
        'epocas_fase1':        len(history.history['accuracy']),
        'epocas_fase2':        len(history_fine.history['accuracy']),
    }
    for i, c in enumerate(class_names):
        resultado[f'precision_{c}'] = precision[i]
        resultado[f'recall_{c}']    = recall[i]
        resultado[f'f1_{c}']        = f1[i]

    df_resultado = pd.DataFrame([resultado])
    if os.path.exists(csv_path):
        df_resultado.to_csv(csv_path, mode='a', header=False, index=False)
    else:
        df_resultado.to_csv(csv_path, index=False)

    # Training curves
    fig_curvas, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].set_title('Accuracy')
    axes[0].plot(history.history['accuracy'], label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Validation')
    axes[0].legend()
    axes[1].set_title('Loss')
    axes[1].plot(history.history['loss'], label='Train')
    axes[1].plot(history.history['val_loss'], label='Validation')
    axes[1].legend()
    fig_curvas.tight_layout()
    fig_to_image(fig_curvas, path_curvas)
    plt.close(fig_curvas)

    # Confusion matrix
    fig_cm, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    ax.set_title('Confusion Matrix - Test')
    fig_cm.tight_layout()
    fig_to_image(fig_cm, path_cm)
    plt.close(fig_cm)

    # PDF
    pdf_path = os.path.join(output_dir, 'relatorio.pdf')
    doc = SimpleDocTemplate(pdf_path, pagesize=A4,
                            rightMargin=2*rl_cm, leftMargin=2*rl_cm,
                            topMargin=2*rl_cm, bottomMargin=2*rl_cm)

    styles = getSampleStyleSheet()
    style_title = ParagraphStyle('title', fontSize=18, alignment=TA_CENTER, spaceAfter=10, fontName='Helvetica-Bold')
    style_h2    = ParagraphStyle('h2', fontSize=13, spaceAfter=6, spaceBefore=12, fontName='Helvetica-Bold')
    style_body  = styles['Normal']

    story = []

    story.append(Paragraph('Training report', style_title))
    story.append(Paragraph(f'Configuration: {config_name}', style_body))
    story.append(Paragraph(f'Date: {datetime.datetime.now().strftime("%d/%m/%Y %H:%M")}', style_body))
    story.append(Spacer(1, 0.5*rl_cm))

    story.append(Paragraph('Hyperparameters', style_h2))
    dados_hiper = [
        ['Parameter', 'Value'],
        ['Input size',         f'{img_size[0]}x{img_size[1]}'],
        ['Batch size',         str(batch_size)],
        ['Phase 1 - Epochs',      str(len(history.history['accuracy']))],
        ['Phase 2 - Epochs',      str(len(history_fine.history['accuracy']))],
        ['Samples per class', str(samples_per_class)],
    ]
    t = Table(dados_hiper, colWidths=[8*rl_cm, 8*rl_cm])
    t.setStyle(TableStyle([
        ('BACKGROUND',    (0,0), (-1,0), colors.HexColor('#2c3e50')),
        ('TEXTCOLOR',     (0,0), (-1,0), colors.white),
        ('FONTNAME',      (0,0), (-1,0), 'Helvetica-Bold'),
        ('ALIGN',         (0,0), (-1,-1), 'CENTER'),
        ('GRID',          (0,0), (-1,-1), 0.5, colors.grey),
        ('ROWBACKGROUNDS',(0,1), (-1,-1), [colors.white, colors.HexColor('#f2f2f2')]),
    ]))
    story.append(t)
    story.append(Spacer(1, 0.5*rl_cm))

    story.append(Paragraph('General Metrics', style_h2))
    dados_metricas = [
        ['Metric', 'Value'],
        ['Global accuracy',      f'{acc:.4f}'],
        ['Balanced Accuracy',    f'{bal_acc:.4f}'],
        ['Test Loss',         f'{loss:.4f}'],
        ['Average Inference Time', f'{np.mean(tempos_ms):.2f} ms'],
    ]
    t2 = Table(dados_metricas, colWidths=[8*rl_cm, 8*rl_cm])
    t2.setStyle(TableStyle([
        ('BACKGROUND',    (0,0), (-1,0), colors.HexColor('#2c3e50')),
        ('TEXTCOLOR',     (0,0), (-1,0), colors.white),
        ('FONTNAME',      (0,0), (-1,0), 'Helvetica-Bold'),
        ('ALIGN',         (0,0), (-1,-1), 'CENTER'),
        ('GRID',          (0,0), (-1,-1), 0.5, colors.grey),
        ('ROWBACKGROUNDS',(0,1), (-1,-1), [colors.white, colors.HexColor('#f2f2f2')]),
    ]))
    story.append(t2)
    story.append(Spacer(1, 0.5*rl_cm))

    story.append(Paragraph('Metrics per Class', style_h2))
    dados_classes = [['Class', 'Precision', 'Recall', 'F1', 'Support']]
    for i, c in enumerate(class_names):
        dados_classes.append([c, f'{precision[i]:.4f}', f'{recall[i]:.4f}', f'{f1[i]:.4f}', str(support[i])])
    t3 = Table(dados_classes, colWidths=[4*rl_cm, 4*rl_cm, 4*rl_cm, 4*rl_cm, 4*rl_cm])
    t3.setStyle(TableStyle([
        ('BACKGROUND',    (0,0), (-1,0), colors.HexColor('#2c3e50')),
        ('TEXTCOLOR',     (0,0), (-1,0), colors.white),
        ('FONTNAME',      (0,0), (-1,0), 'Helvetica-Bold'),
        ('ALIGN',         (0,0), (-1,-1), 'CENTER'),
        ('GRID',          (0,0), (-1,-1), 0.5, colors.grey),
        ('ROWBACKGROUNDS',(0,1), (-1,-1), [colors.white, colors.HexColor('#f2f2f2')]),
    ]))
    story.append(t3)
    story.append(Spacer(1, 0.5*rl_cm))

    story.append(Paragraph('Training Curves', style_h2))
    story.append(Image(path_curvas, width=16*rl_cm, height=6*rl_cm))
    story.append(Spacer(1, 0.5*rl_cm))

    story.append(Paragraph('Confusion Matrix', style_h2))
    story.append(Image(path_cm, width=12*rl_cm, height=9*rl_cm))

    doc.build(story)
    os.remove(path_curvas)
    os.remove(path_cm)

    print(f'Report successfully saved at {pdf_path}')
    print(f'CSV results successfully saved at {csv_path}')

Writing report.py


### 2. Architecture and Training Pipeline

This section defines the core deep learning pipeline of the proposed research, addressing the severe class imbalance and edge-hardware constraints discussed in the manuscript. The implementation is divided into the following sequential stages:

* **Temporal Out-of-Time Split:** The dataset is strictly divided using 2020 data exclusively for testing, while other years are split into training (60%), validation (15%), and conformal calibration (25%) sets to prevent temporal data leakage.
* **Dynamic Augmentation:** An Albumentations pipeline introduces class-conditional transformations. Critical overflow events (*high* and *flood*) receive aggressive environmental distortions (e.g., random rain, fog, and shadows) to simulate adverse weather conditions and mitigate spatial overfitting. The amount of samples classified as low were also upsampled by 10x the default sample value (`SAMPLES_PER_CLASS=200`), while the images from the others classes remained unchanged.
* **Semantic Masking:** An image has a 50% chance of being masked, darkening the background and isolating the water, and a 50% probability of remaining unchanged.
* **Sparse Categorical Focal Loss:** Standard cross-entropy is replaced by a custom focal loss function ($\gamma = 2.0$) dynamically weighted by Scikit-Learn's class distribution matrices. This mathematical framework suppresses the gradient contribution of easily classified majority samples (normal water levels).
* **Two-Phase Fine-Tuning:** The pipeline leverages a pre-trained `MobileNetV3Large` architecture. Phase 1 trains the custom classification head while freezing the base model. Phase 2 unfreezes the top 50 layers for fine-grained feature extraction, utilizing adaptive learning rate scheduling and early stopping to prevent over-parameterization.

In [3]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tensorflow as tf
import time
import math
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
import tensorflow.keras.backend as K
from report import gerar_relatorio
import albumentations as A

CONFIG_NAME        = 'mobilenetv3_224x224'
DATASET_IMAGENS    = '/kaggle/input/datasets/caetanoranieri/river-images-at-sao-carlos/enoe/enoe2' 
DATASET_MASCARAS   = '/kaggle/input/datasets/guilhermebisse/sao-carlos-river-masks-full-batch'
CSV_PATH           = '/kaggle/input/datasets/caetanoranieri/river-images-at-sao-carlos/flood_images_annot_2.csv'
OUTPUT_DIR         = f'/kaggle/working/{CONFIG_NAME}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(CSV_PATH, index_col=0)
CLASS_NAMES  = ['low', 'medium', 'high', 'flood']
class_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}

df['label']     = df['level'].map(class_to_idx)
df['full_path'] = df['path'].apply(lambda p: os.path.join(DATASET_IMAGENS, p))
df['mask_path'] = df['path'].apply(lambda p: os.path.join(DATASET_MASCARAS, p.replace('.jpg', '_mask.png')))

IMG_SIZE          = (224, 224)
BATCH_SIZE        = 32
SAMPLES_PER_CLASS = 200

ano_teste = 2020
df['year'] = pd.to_datetime(df['datetime']).dt.year

df_test      = df[df['year'] == ano_teste].copy()
df_train_val = df[df['year'] != ano_teste].copy()

# Temporal Out-of-Time Split (Train 60%, Val 15%, Calib 25%)
df_train_temp, df_val = train_test_split(df_train_val, test_size=0.15, random_state=42, stratify=df_train_val['label'])
df_train, df_calib   = train_test_split(df_train_temp, test_size=0.2941, random_state=42, stratify=df_train_temp['label'])

df_low    = df_train[df_train['label'] == 0]
df_medium = df_train[df_train['label'] == 1]
df_high   = df_train[df_train['label'] == 2]
df_flood  = df_train[df_train['label'] == 3]

def load_image_simples(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3, dct_method='INTEGER_ACCURATE')
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.clip_by_value(img, 0, 255)
    return img, label

def make_dataset(df_split, shuffle=False):
    paths  = df_split['full_path'].values
    labels = df_split['label'].values.astype('int32')
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image_simples, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=1000)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

def aplicar_efeito_mascara(img, mask_path):
    if not mask_path or not os.path.exists(mask_path):
        return img
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return img
    
    mask = cv2.resize(mask, IMG_SIZE)
    _, mask_bin = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    mask_inv = cv2.bitwise_not(mask_bin)

    agua = cv2.bitwise_and(img, img, mask=mask_bin)
    cenario = cv2.bitwise_and(img, img, mask=mask_inv)
    cenario_escuro = cv2.convertScaleAbs(cenario, alpha=0.4, beta=-20)
    return cv2.add(agua, cenario_escuro)

class ResampleGenerator(tf.keras.utils.PyDataset):
    def __init__(self, df_low, df_medium, df_high, df_flood, batch_size, **kwargs):
        super().__init__(**kwargs)
        self.df_low     = df_low
        self.df_medium  = df_medium
        self.df_high    = df_high
        self.df_flood   = df_flood
        self.batch_size = batch_size
        self.epoch      = 0
        
        self.pipeline_comum = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=8, border_mode=cv2.BORDER_REFLECT, p=0.3),
            A.RandomBrightnessContrast(0.07, 0.07, p=0.3),
            A.GaussianBlur(blur_limit=(3, 5), p=0.15),
        ])
        self.pipeline_critico = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=10, border_mode=cv2.BORDER_REFLECT, p=0.4),
            A.RandomBrightnessContrast(0.15, 0.15, p=0.5),
            A.RandomRain(p=0.35),
            A.RandomFog(fog_coef_range=(0.1, 0.25), p=0.35),
            A.GaussianBlur(blur_limit=(3, 5), p=0.30),
            A.RandomShadow(p=0.30),
        ])
        self._resample()

    def _resample(self):
        df_low_down  = resample(self.df_low,    replace=False, n_samples=SAMPLES_PER_CLASS*10, random_state=self.epoch)
        df_medium_up = resample(self.df_medium, replace=True,  n_samples=SAMPLES_PER_CLASS, random_state=self.epoch)
        df_high_up   = resample(self.df_high,   replace=True,  n_samples=SAMPLES_PER_CLASS, random_state=self.epoch)
        df_flood_up  = resample(self.df_flood,  replace=True,  n_samples=SAMPLES_PER_CLASS, random_state=self.epoch)
        self.df_current = pd.concat([df_low_down, df_medium_up, df_high_up, df_flood_up]).sample(frac=1, random_state=self.epoch).reset_index(drop=True)

    def __len__(self):
        return math.ceil(len(self.df_current) / self.batch_size)

    def __getitem__(self, idx):
        batch = self.df_current.iloc[idx * self.batch_size:(idx + 1) * self.batch_size]
        images, labels = [], []

        for _, row in batch.iterrows():
            img = cv2.imread(row['full_path'])
            if img is None:
                img = np.zeros((*IMG_SIZE, 3), dtype=np.uint8)
            else:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, IMG_SIZE)

            if np.random.random() < 0.5:
                img = aplicar_efeito_mascara(img, row['mask_path'])
            if row["label"] == 0 or row["label"] == 1:
                transformed = self.pipeline_comum(image=img)
                img = transformed["image"]
            else:
                transformed = self.pipeline_critico(image=img)
                img = transformed["image"]
            images.append(img)
            labels.append(row["label"])

        return np.array(images), np.array(labels)

    def on_epoch_end(self):
        self.epoch += 1
        self._resample()

def sparse_categorical_focal_loss(gamma=2.0, alpha_per_class=[1.0, 1.0, 1.0, 1.0]):
    alpha_tensor = tf.constant(alpha_per_class, dtype=tf.float32)
    
    def loss(y_true, y_pred):
        # Ensure strict 1D shape alignment to maintain computational graph stability
        y_true = tf.cast(y_true, tf.int32)
        y_true = tf.reshape(y_true, [-1]) 
        
        # Stabilize predictions to prevent NaN during log computations
        y_pred = tf.clip_by_value(y_pred, K.epsilon(), 1.0 - K.epsilon())
        
        # 1. Standard Categorical Cross-Entropy
        ce = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
        
        # 2. Extract confidence for the true class (p_t)
        one_hot_labels = tf.one_hot(y_true, depth=tf.shape(y_pred)[-1])
        p_t = tf.reduce_sum(one_hot_labels * y_pred, axis=-1)
        
        # 3. Dynamic Alpha weighting
        alpha_t = tf.gather(alpha_tensor, y_true)
        
        # 4. Focal Loss formulation
        return alpha_t * tf.pow(1.0 - p_t, gamma) * ce
        
    return loss

def visualizar_augmentacao(df_high, df_flood, pipeline):
    n_amostras = 2

    classes = [(df_high, "High"), (df_flood, "Flood")]
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))

    for row_idx, (df_classe, classe_nome) in enumerate(classes):
        
        amostras = df_classe.sample(n_amostras, random_state=42)

        for i, (_, row) in enumerate(amostras.iterrows()):

            img = cv2.imread(row["full_path"])
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, IMG_SIZE)

            img_aug = pipeline(image=img)["image"]

            col_orig = i * 2
            col_aug = i * 2 + 1

            axes[row_idx, col_orig].imshow(img)
            axes[row_idx, col_orig].axis("off")
            axes[row_idx, col_orig].set_title(
                f"Sample {i+1} - {classe_nome}\nBefore",
                fontsize=11
            )

            axes[row_idx, col_aug].imshow(img_aug)
            axes[row_idx, col_aug].axis("off")
            axes[row_idx, col_aug].set_title(
                f"Sample {i+1} - {classe_nome}\nAfter",
                fontsize=11
            )

            axes[row_idx, col_orig].text(
                1.08,
                0.5,
                "→",
                fontsize=28,
                fontweight="bold",
                transform=axes[row_idx, col_orig].transAxes,
                ha="center",
                va="center"
            )

        axes[row_idx, 0].set_ylabel(
            classe_nome.upper(),
            fontsize=14,
            fontweight="bold",
            rotation=90,
            labelpad=20
        )

    plt.suptitle(
        "Examples of Data Augmentation",
        fontsize=18,
        fontweight="bold"
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(OUTPUT_DIR, "aug_critico.png"),
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close()


train_gen = ResampleGenerator(df_low, df_medium, df_high, df_flood, BATCH_SIZE)
val_ds    = make_dataset(df_val)
test_ds   = make_dataset(df_test)
calib_ds  = make_dataset(df_calib)

pesos_seguros = compute_class_weight(
    class_weight='balanced', 
    classes=np.unique(train_gen.df_current['label']), 
    y=train_gen.df_current['label'].values
)
alpha_custom = list(pesos_seguros)
print(f"Calculated Alphas for Focal Loss: {alpha_custom}")

visualizar_augmentacao(df_high, df_flood, train_gen.pipeline_critico)

# Architecture initialization and training parameters
base_model = MobileNetV3Large(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = preprocess_input(inputs)
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu', kernel_regularizer=l2(1e-4))(x)
x = Dropout(0.4)(x)
predictions = Dense(4, activation='softmax')(x)

model = Model(inputs=inputs, outputs=predictions)
model.compile(
    optimizer=Adam(learning_rate=1e-5), 
    loss=sparse_categorical_focal_loss(gamma=2.0, alpha_per_class=alpha_custom),
    metrics=['accuracy']
)

# PHASE 1: Feature Extraction (Frozen Backbone)
print("Starting Phase 1: Feature Extraction")
history = model.fit(
    train_gen,
    validation_data=val_ds,
    epochs=100,
    callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)]
)

# PHASE 2: Fine-Tuning (Unfreezing 50 last layers)
print("Starting Phase 2: Fine-Tuning")
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=5e-6), 
    loss=sparse_categorical_focal_loss(gamma=2.0, alpha_per_class=alpha_custom),
    metrics=['accuracy']
)

early_stopping_final = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=7, min_lr=1e-7)

history_fine = model.fit(
    train_gen, 
    validation_data=val_ds, 
    epochs=100,
    callbacks=[early_stopping_final, lr_scheduler]
)

2026-08-20 19:06:30.805500: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Calculated Alphas for Focal Loss: [np.float64(0.325), np.float64(3.25), np.float64(3.25), np.float64(3.25)]
12683000/12683000 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Starting Phase 1: Feature Extraction
Epoch 1/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.2354 - loss: 1.3163

Corrupt JPEG data: 32764 extraneous bytes before marker 0xd9


82/82 ━━━━━━━━━━━━━━━━━━━━ 353s 4s/step - accuracy: 0.2800 - loss: 1.2339 - val_accuracy: 0.5615 - val_loss: 0.2467
Epoch 2/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.3992 - loss: 0.9472

Corrupt JPEG data: 32764 extraneous bytes before marker 0xd9


82/82 ━━━━━━━━━━━━━━━━━━━━ 321s 4s/step - accuracy: 0.4069 - loss: 1.0061 - val_accuracy: 0.6638 - val_loss: 0.2106
Epoch 3/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4237 - loss: 0.8779

... (274 outputs left out)

Corrupt JPEG data: 32764 extraneous bytes before marker 0xd9


82/82 ━━━━━━━━━━━━━━━━━━━━ 318s 4s/step - accuracy: 0.9008 - loss: 0.1969 - val_accuracy: 0.9725 - val_loss: 0.0820 - learning_rate: 1.0000e-06
Epoch 11/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8777 - loss: 0.2326

Corrupt JPEG data: 32764 extraneous bytes before marker 0xd9


82/82 ━━━━━━━━━━━━━━━━━━━━ 309s 4s/step - accuracy: 0.8815 - loss: 0.2303 - val_accuracy: 0.9714 - val_loss: 0.0824 - learning_rate: 1.0000e-06


### 3. Out-of-Time Evaluation and Conformal Prediction Calibration

This final section evaluates the trained pipeline on the unseen 2020 test dataset and applies the Class-Conditional Conformal Prediction (CP) framework to mathematically bound predictive uncertainty. The process executes the following steps:

* **Inference and Calibration:** The model computes softmax probabilities for both the calibration partition and the out-of-time test set.
* **Dynamic Thresholding:** Using a predefined error tolerance of $\alpha = 0.05$, the CP framework calculates strict class-conditional quantiles. This establishes independent minimum confidence thresholds for each water level state, adjusting for class difficulty.
* **Risk-Averse Fallback:** Softmax outputs are transformed into sets of valid classes. If severe environmental noise prevents any class from reaching its required threshold (yielding an empty set $\emptyset$), the pipeline actively intercepts the prediction and systematically defaults to the conservative *argmax* transitional state.
* **Reporting and Visualization:** Final deterministic metrics (post-fallback) are evaluated and compiled into the automated PDF report. Finally, an individual inference sample is plotted to visually demonstrate the conformal prediction behavior and threshold boundaries.

In [4]:
tempos_ms = []
for img, _ in test_ds.unbatch().take(200):
    start = time.time()
    _ = model(np.expand_dims(img.numpy(), axis=0), training=False)
    tempos_ms.append((time.time() - start) * 1000)

loss, acc = model.evaluate(test_ds)

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_probs = model.predict(test_ds)

calib_probs = model.predict(calib_ds)
calib_true = np.concatenate([y.numpy() for _, y in calib_ds])

def predict_with_class_conditional_conformal(probs, calib_probs, calib_true, alpha=0.10):
    n_classes = probs.shape[1]
    qhats = np.zeros(n_classes)
    prediction_sets = np.zeros_like(probs, dtype=bool)

    
    for c in range(n_classes):
        idx_c = np.where(calib_true == c)[0]
        n_c = len(idx_c)
        
        if n_c == 0:
            qhats[c] = 1.0
            continue
            
        # Calculate the error scores for this class only
        calib_scores_c = 1 - calib_probs[idx_c, c]
        
        # Calculate the specific quantile for this class
        q_level_c = np.ceil((n_c + 1) * (1 - alpha)) / n_c
        q_level_c = min(q_level_c, 1.0) 
        
        qhats[c] = np.quantile(calib_scores_c, q_level_c, method='higher')
        
        # Applies the threshold (1 - qhat) only to the probability column for this class
        prediction_sets[:, c] = (probs[:, c] >= (1 - qhats[c]))
        
    return prediction_sets, qhats

alpha_value = 0.05
prediction_sets, qhats = predict_with_class_conditional_conformal(y_probs, calib_probs, calib_true, alpha=alpha_value)

np.save(os.path.join(OUTPUT_DIR, 'conformal_qhats.npy'), qhats)
np.save(os.path.join(OUTPUT_DIR, 'conformal_thresholds.npy'), 1 - qhats)
print(f"Conformal thresholds saved at {OUTPUT_DIR}")

cobertura = np.mean(prediction_sets[np.arange(len(y_true)), y_true])
tamanho_medio = np.mean(np.sum(prediction_sets, axis=1))

y_pred_argmax = np.argmax(y_probs, axis=1)
y_pred_conformal = []

for row_set, prob_vec, argmax_val in zip(prediction_sets, y_probs, y_pred_argmax):
    indices_ativos = np.where(row_set)[0]
    
    if len(indices_ativos) > 0:
        # Choose the best option only from among the mathematically valid classes
        best_idx = indices_ativos[np.argmax(prob_vec[indices_ativos])]
        y_pred_conformal.append(best_idx)
    else:
        y_pred_conformal.append(argmax_val)

y_pred_conformal = np.array(y_pred_conformal)

print(f"\n--- Conformal Prediction metrics ---")
print(f"Alpha: {alpha_value}")
print(f"Coverage (Statistical Guarantee): {cobertura:.2%}")
print(f"Average Ambiguity (Set Size): {tamanho_medio:.2f} classes")

limiares_exigidos = 1 - qhats
print(f"Minimum Thresholds of Certainty Required by Class:")
for i, nome in enumerate(CLASS_NAMES):
    print(f" - {nome}: {limiares_exigidos[i]:.2%}")

from sklearn.metrics import confusion_matrix as sk_confusion_matrix
cm_final_report = sk_confusion_matrix(y_true, y_pred_conformal, labels=[0, 1, 2, 3])

gerar_relatorio(
    config_name=CONFIG_NAME, output_dir=OUTPUT_DIR, class_names=CLASS_NAMES,
    acc=acc, loss=loss, tempos_ms=tempos_ms,
    y_true=y_true, y_pred=y_pred_conformal, cm=cm_final_report,
    history=history, history_fine=history_fine,
    img_size=IMG_SIZE, batch_size=BATCH_SIZE, samples_per_class=SAMPLES_PER_CLASS,
)

# CONFORMAL PREDICTION BEHAVIOR DISPLAY
for img, _ in test_ds.unbatch().take(1):
    break
    
plt.figure(figsize=(18, 5))

img_np = img.numpy()
img_np = img_np.astype(np.uint8)

if img_np.min() < 0:
    img_np = (img_np + 1) / 2.0

true_cls = y_true[0]
final_pred = y_pred_conformal[0]

classes_no_set = np.where(prediction_sets[0])[0]
nomes_no_set = [CLASS_NAMES[c] for c in classes_no_set]

if len(nomes_no_set) == 0:
    set_text = "Empty set (Fallback to High class)"
else:
    set_text = ", ".join(nomes_no_set)

cor_titulo = 'green' if final_pred == true_cls else 'red'

plt.subplot(1, 2, 1)
plt.imshow(img_np)
plt.axis('off')
plt.title(f"TRUE: {CLASS_NAMES[true_cls]} | PREDICTION: {CLASS_NAMES[final_pred]}", color=cor_titulo, fontsize=12, fontweight='bold')
        
plt.subplot(1, 2, 2)

plt.tick_params(axis='both', labelsize=14)
bars = plt.barh(CLASS_NAMES, y_probs[0], color='lightgray', edgecolor='black', height=0.5)
        
for idx_barra, bar in enumerate(bars):
    if idx_barra in classes_no_set:
        bar.set_color('#4CAF50')
    if idx_barra == final_pred:
        bar.set_edgecolor('blue')
        bar.set_linewidth(2)
                
for c_idx in range(len(CLASS_NAMES)):
    limiar = 1 - qhats[c_idx]
    plt.vlines(x=limiar, ymin=c_idx-0.25, ymax=c_idx+0.25, colors='red', linestyles='dashed', alpha=0.7)
    
plt.xlim(0, 1.05)
plt.xlabel("Probability / Confidence", fontsize=14)
plt.title(f"Classes in the Conformal Set: [{set_text}]", fontsize=16)
        
plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, 'conformal_visualizacao.png')
plt.savefig(plot_path, dpi=150)
print(f"Visualization saved at: {plot_path}")
plt.show()


model.save(os.path.join(OUTPUT_DIR, 'flood_model.keras'))

335/501 ━━━━━━━━━━━━━━━━━━━━ 2:05 754ms/step - accuracy: 0.9294 - loss: 0.0958

Corrupt JPEG data: 32764 extraneous bytes before marker 0xd9


483/501 ━━━━━━━━━━━━━━━━━━━━ 13s 747ms/step - accuracy: 0.9392 - loss: 0.0922

Corrupt JPEG data: 32767 extraneous bytes before marker 0xd9


501/501 ━━━━━━━━━━━━━━━━━━━━ 373s 743ms/step - accuracy: 0.9591 - loss: 0.0861


Corrupt JPEG data: 32764 extraneous bytes before marker 0xd9
Corrupt JPEG data: 32767 extraneous bytes before marker 0xd9


... (5 outputs left out)

Corrupt JPEG data: 32763 extraneous bytes before marker 0xd9


411/411 ━━━━━━━━━━━━━━━━━━━━ 301s 730ms/step


Corrupt JPEG data: 32763 extraneous bytes before marker 0xd9


Conformal thresholds saved at /kaggle/working/mobilenetv3_224x224

--- Conformal Prediction metrics ---
Alpha: 0.05
Coverage (Statistical Guarantee): 89.62%
Average Ambiguity (Set Size): 1.00 classes
Minimum Thresholds of Certainty Required by Class:
 - low: 70.25%
 - medium: 34.88%
 - high: 5.44%
 - flood: 51.39%
Report successfully saved at /kaggle/working/mobilenetv3_224x224/relatorio.pdf
CSV results successfully saved at /kaggle/working/resultados.csv
Visualization saved at: /kaggle/working/mobilenetv3_224x224/conformal_visualizacao.png
